In [1]:
import os
import glob
import pydicom
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import kagglehub

# ==========================================
# 1. CONFIGURATION & ENVIRONMENT SETUP
# ==========================================
DATA_DIR = '/kaggle/input/rsna-knee-abnormality-detection'
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
TEST_SERIES_CSV = os.path.join(DATA_DIR, 'test_series.csv')
SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission.csv')
TEST_SERIES_DIR = os.path.join(DATA_DIR, 'test_series')
OUTPUT_SUB = 'submission.csv'

# 12 Abnormality Target Labels
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

# Hardware Acceleration Setup (GPU T4 x2)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SLICES_PER_PLANE = 5  # Number of key representative slices per plane

print(f"Active Compute Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Available GPUs: {torch.cuda.device_count()} ({torch.cuda.get_device_name(0)})")

# Standard Normalization Pipeline for DINOv2
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 2. DICOM PROCESSING & DATASET
# ==========================================
def read_dicom_slice(fpath):
    """Reads a DICOM file, applies min-max scaling, and converts it to an RGB tensor."""
    try:
        dicom = pydicom.dcmread(fpath)
        img = dicom.pixel_array.astype(np.float32)
        if img.max() > img.min():
            img = (img - img.min()) / (img.max() - img.min())
        else:
            img = np.zeros_like(img)
        img = (img * 255).astype(np.uint8)
        pil_img = Image.fromarray(img).convert('RGB')
        return transform(pil_img)
    except Exception:
        return torch.zeros((3, 224, 224), dtype=torch.float32)

class RSNADINOv2TestDataset(Dataset):
    def __init__(self, test_df, series_df):
        self.uids = test_df['StudyInstanceUID'].values
        self.series_df = series_df

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, idx):
        study_uid = self.uids[idx]
        study_series = self.series_df[self.series_df['StudyInstanceUID'] == study_uid]
        
        planes = ['Sagittal', 'Coronal', 'Axial']
        all_planes_tensors = []

        for plane in planes:
            plane_match = study_series[study_series['Anatomical_Plane'] == plane]
            slice_tensors = []
            
            if not plane_match.empty:
                series_uid = plane_match.iloc[0]['SeriesInstanceUID']
                s_path = os.path.join(TEST_SERIES_DIR, study_uid, series_uid)
                dcm_files = sorted(glob.glob(os.path.join(s_path, "*.dcm")))
                
                if dcm_files:
                    indices = np.linspace(0, len(dcm_files) - 1, SLICES_PER_PLANE, dtype=int)
                    for i in indices:
                        slice_tensors.append(read_dicom_slice(dcm_files[i]))
            
            while len(slice_tensors) < SLICES_PER_PLANE:
                slice_tensors.append(torch.zeros((3, 224, 224), dtype=torch.float32))
                
            all_planes_tensors.append(torch.stack(slice_tensors, dim=0))

        return study_uid, torch.stack(all_planes_tensors, dim=0)

# ==========================================
# 3. DINOv2 CHANNELS ATTENTION MODEL
# ==========================================
class DINOv2KneeClassifier(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()
        # Load DINOv2 ViT-Small via PyTorch Hub
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', source='github', pretrained=False)
        embed_dim = 384
        
        self.attn_pool = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
            nn.Softmax(dim=1)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, p, s, c, h, w = x.shape
        x = x.view(b * p * s, c, h, w)
        
        features = self.backbone(x)
        features = features.view(b * p, s, -1)
        
        weights = self.attn_pool(features)
        pooled_feat = torch.sum(features * weights, dim=1)
        pooled_feat = pooled_feat.view(b, p * 384)
        
        return self.classifier(pooled_feat)

# ==========================================
# 4. INFERENCE EXECUTION PIPELINE
# ==========================================
def run_pipeline():
    print("Starting pipeline execution...")
    
    # Check if test set exists (Interactive vs Submission mode)
    if not (os.path.exists(TEST_CSV) and os.path.exists(TEST_SERIES_CSV)):
        print("Test set not detected (Interactive Mode). Generating dummy submission.csv...")
        if os.path.exists(SAMPLE_SUB):
            sub_df = pd.read_csv(SAMPLE_SUB)
        else:
            # Fallback dummy dataframe creation
            dummy_data = {'StudyInstanceUID': ['dummy_1', 'dummy_2']}
            for col in TARGET_COLS:
                dummy_data[col] = [0.5, 0.5]
            sub_df = pd.DataFrame(dummy_data)
            
        sub_df.to_csv(OUTPUT_SUB, index=False)
        print(f"File created successfully at {OUTPUT_SUB}")
        return

    test_df = pd.read_csv(TEST_CSV)
    series_df = pd.read_csv(TEST_SERIES_CSV)

    dataset = RSNADINOv2TestDataset(test_df, series_df)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=2)

    base_model = DINOv2KneeClassifier(num_classes=12)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(base_model)
    else:
        model = base_model

    model.to(DEVICE)
    model.eval()

    results = []
    print("Running inference with Test-Time Augmentation (TTA)...")

    with torch.no_grad():
        for study_uids, volumes in dataloader:
            volumes = volumes.to(DEVICE)
            
            # Predict
            preds = model(volumes)
            
            # TTA: Horizontal Flip
            volumes_flipped = torch.flip(volumes, dims=[-1])
            preds_flipped = model(volumes_flipped)
            
            # Blend
            final_preds = ((preds + preds_flipped) / 2.0).cpu().numpy()

            for uid, pred in zip(study_uids, final_preds):
                row = {'StudyInstanceUID': uid}
                for i, col in enumerate(TARGET_COLS):
                    row[col] = float(pred[i])
                results.append(row)

    sub_df = pd.DataFrame(results)
    sub_df = sub_df[['StudyInstanceUID'] + TARGET_COLS]
    sub_df.to_csv(OUTPUT_SUB, index=False)
    print(f"Inference complete. File written to {OUTPUT_SUB}")

# Direct execution call for Jupyter/Kaggle environments
run_pipeline()

Active Compute Device: cuda
Available GPUs: 2 (Tesla T4)
Starting pipeline execution...
Test set not detected (Interactive Mode). Generating dummy submission.csv...
File created successfully at submission.csv


In [2]:
import os
import pandas as pd

# Monitor Script for Submission Verification
SUB_FILE = 'submission.csv'

print("=" * 50)
print(" SUBMISSION FILE MONITOR ")
print("=" * 50)

if os.path.exists(SUB_FILE):
    # File statistics
    file_size_kb = os.path.getsize(SUB_FILE) / 1024
    sub_df = pd.read_csv(SUB_FILE)
    
    print(f" Status: File Found!")
    print(f" File Name: {SUB_FILE}")
    print(f" File Size: {file_size_kb:.2f} KB")
    print(f" Total Rows: {len(sub_df)}")
    print(f" Total Columns: {len(sub_df.columns)}")
    print("-" * 50)
    
    # Missing value check
    null_counts = sub_df.isnull().sum().sum()
    if null_counts == 0:
        print(" Data Quality Check: PASSED (No NaN/null values found)")
    else:
        print(f" Data Quality Check: WARNING! ({null_counts} NaN values detected)")
        
    print("-" * 50)
    print(" First 5 Rows Preview:")
    display(sub_df.head())
else:
    print(f" Status: File NOT Found! ({SUB_FILE} has not been generated yet.)")

print("=" * 50)

 SUBMISSION FILE MONITOR 
 Status: File Found!
 File Name: submission.csv
 File Size: 0.24 KB
 Total Rows: 2
 Total Columns: 13
--------------------------------------------------
 Data Quality Check: PASSED (No NaN/null values found)
--------------------------------------------------
 First 5 Rows Preview:


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,dummy_1,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,dummy_2,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
